## Shrinkage Experiment 2: Understanding When Shrinkage Methods Help

## 1. Objective : 
In the previous experiment, OLS, Ridge, and Lasso produced very similar results.

This happened because the dataset was relatively easy:

- Large number of observations
- Small number of features
- Strong signal
- Low risk of overfitting

As a result, OLS was already able to generalize well, leaving little room for regularization methods to demonstrate their benefits.

---

### 2. Why This Experiment?

The goal of this experiment is to create a situation where OLS is likely to struggle.

We want to intentionally build a dataset that is:

- High-dimensional (many features)
- Small in sample size
- Mostly noise
- Prone to overfitting

This will help us observe how shrinkage methods behave when the learning problem becomes difficult.

---



### 3. Problem Statement

Suppose we are building a model to predict house prices.

For each house, we have collected 100 features.

Examples might include:

- House size
- Number of bedrooms
- Location score
- Garage information
- Paint color
- House number
- WiFi password length
- Garden decorations
- Various other recorded attributes

However, in reality only a few of these features truly influence house prices.

Most of the collected variables are either irrelevant or contain only random noise.

---

### 4. Hidden Reality

The true price of a house depends on only three features:

```text
Price = 5*x1 - 3*x2 + 8*x3 + random noise
```

where:

- x1 = house size
- x2 = number of bedrooms
- x3 = location quality

All remaining features:

```text
x4, x5, ..., x100
```

have absolutely no effect on price.

These are pure noise variables.

---

### 5. Dataset Design

We intentionally create:

```text
Number of observations (n) = 40

Number of features (p) = 100
```

This means:

```text
40 samples
100 predictors
```

The model now has access to far more features than can be reliably supported by the available data.

This creates a high-risk overfitting scenario.

---



## What We Expect to Observe

## - OLS Regression

OLS tries to minimize training error only.

Because there are many noisy variables available, OLS may start assigning non-zero coefficients to random patterns that exist only in the training data.

Expected behavior:

- Very low training error
- Unstable coefficients
- Worse generalization
- Many noise variables receive non-zero weights

---

## - Ridge Regression

Ridge introduces an L2 penalty:

```text
Loss = Training Error + λΣβ²
```

Expected behavior:

- Coefficients shrink toward zero
- Less sensitivity to noise
- More stable model
- Improved generalization

---

## - Lasso Regression

Lasso introduces an L1 penalty:

```text
Loss = Training Error + λΣ|β|
```

Expected behavior:

- Coefficients shrink toward zero
- Many noise variables become exactly zero
- Automatic feature selection
- Simpler and more interpretable model

---

###  Questions We Want to Answer

1. Does OLS assign large coefficients to noise variables?

2. How much do Ridge and Lasso shrink these coefficients?

3. Does Lasso successfully identify the truly important features?

4. How do training and test errors compare across the models?

5. When the number of predictors greatly exceeds the amount of available information, do shrinkage methods generalize better than OLS?

---


### Step 1: Create Synthetic dataset

In [55]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 40 # samples = houses
p = 100  # features

#genetare random samples with 100 features
x = np.random.randn(n , p)


**- Converting X into a datframe:**

In [95]:


#name the columns
columns = []

for i in range(p):
    columns.append(f"feature_{i + 1}")

#dataframe
df = pd.DataFrame(
    x,
    columns = columns,
    
)

df.head()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_91,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100
0,0.496714,-0.138264,0.647689,1.523030,-0.234153,-0.234137,1.579213,0.767435,-0.469474,0.542560,...,0.097078,0.968645,-0.702053,-0.327662,-0.392108,-1.463515,0.296120,0.261055,0.005113,-0.234587
1,-1.415371,-0.420645,-0.342715,-0.802277,-0.161286,0.404051,1.886186,0.174578,0.257550,-0.074446,...,-0.446515,0.856399,0.214094,-1.245739,0.173181,0.385317,-0.883857,0.153725,0.058209,-1.142970
2,0.357787,0.560785,1.083051,1.053802,-1.377669,-0.937825,0.515035,0.513786,0.515048,3.852731,...,-0.208122,-0.493001,-0.589365,0.849602,0.357015,-0.692910,0.899600,0.307300,0.812862,0.629629
3,-0.828995,-0.560181,0.747294,0.610370,-0.020902,0.117327,1.277665,-0.591571,0.547097,-0.202193,...,-0.898415,0.491919,-1.320233,1.831459,1.179440,-0.469176,-1.713135,1.353872,-0.114540,1.237816
4,-1.594428,-0.599375,0.005244,0.046981,-0.450065,0.622850,-1.067620,-0.142379,0.120296,0.514439,...,-0.007973,1.479944,0.077368,-0.861284,1.523124,0.538910,-1.037246,-0.190339,-0.875618,-1.382800


In [57]:
print(df.shape)

(40, 100)


In [58]:
# check column names
df.columns

Index(['feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5',
       'feature_6', 'feature_7', 'feature_8', 'feature_9', 'feature_10',
       'feature_11', 'feature_12', 'feature_13', 'feature_14', 'feature_15',
       'feature_16', 'feature_17', 'feature_18', 'feature_19', 'feature_20',
       'feature_21', 'feature_22', 'feature_23', 'feature_24', 'feature_25',
       'feature_26', 'feature_27', 'feature_28', 'feature_29', 'feature_30',
       'feature_31', 'feature_32', 'feature_33', 'feature_34', 'feature_35',
       'feature_36', 'feature_37', 'feature_38', 'feature_39', 'feature_40',
       'feature_41', 'feature_42', 'feature_43', 'feature_44', 'feature_45',
       'feature_46', 'feature_47', 'feature_48', 'feature_49', 'feature_50',
       'feature_51', 'feature_52', 'feature_53', 'feature_54', 'feature_55',
       'feature_56', 'feature_57', 'feature_58', 'feature_59', 'feature_60',
       'feature_61', 'feature_62', 'feature_63', 'feature_64', 'feature_65',
       '

### Step 2: Define the Hidden Truth

We decide: Only the first 3 features affect the house price.
Everything else is useless.

Let's say:

In [61]:
y = (
    5 * df["feature_1"]
    - 3 * df["feature_2"]
    + 8 * df["feature_3"]
    + np.random.normal(0, 2, n)
)

only feature_1, feature_2, feature_3 matter here

**- Adding target to the dataframe**

In [63]:
df['Price'] = y

In [51]:
df.head()

,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_92,feature_93,feature_94,feature_95,feature_96,feature_97,feature_98,feature_99,feature_100,Price
0,0.496714,-0.138264,0.647689,1.523030,-0.234153,-0.234137,1.579213,0.767435,-0.469474,0.542560,...,0.968645,-0.702053,-0.327662,-0.392108,-1.463515,0.296120,0.261055,0.005113,-0.234587,6.864579
1,-1.415371,-0.420645,-0.342715,-0.802277,-0.161286,0.404051,1.886186,0.174578,0.257550,-0.074446,...,0.856399,0.214094,-1.245739,0.173181,0.385317,-0.883857,0.153725,0.058209,-1.142970,-8.831256
2,0.357787,0.560785,1.083051,1.053802,-1.377669,-0.937825,0.515035,0.513786,0.515048,3.852731,...,-0.493001,-0.589365,0.849602,0.357015,-0.692910,0.899600,0.307300,0.812862,0.629629,10.686576
3,-0.828995,-0.560181,0.747294,0.610370,-0.020902,0.117327,1.277665,-0.591571,0.547097,-0.202193,...,0.491919,-1.320233,1.831459,1.179440,-0.469176,-1.713135,1.353872,-0.114540,1.237816,3.929124
4,-1.594428,-0.599375,0.005244,0.046981,-0.450065,0.622850,-1.067620,-0.142379,0.120296,0.514439,...,1.479944,0.077368,-0.861284,1.523124,0.538910,-1.037246,-0.190339,-0.875618,-1.382800,-5.516710


**- Target and features**

In [66]:
X = df.drop(columns = "Price")
y = df["Price"]

### Step 3 : Train and test split

In [67]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split( X , y , test_size = 0.2 , random_state = 42)

In [68]:
print(X_train.shape)
print(X_test.shape)

(32, 100)
(8, 100)


### Step 4: Fit OLS

In [114]:
from sklearn.linear_model import LinearRegression
ols = LinearRegression()
ols.fit(X_train , y_train)

LinearRegression()

**- Inspect ols cofficients**

In [74]:
ols_coef = pd.Series(ols.coef_ , index = X.columns) 
print(ols_coef.sort_values(ascending=False)) 

feature_1     1.699411
feature_3     1.616039
feature_25    0.993106
feature_89    0.880332
feature_32    0.779437
                ...   
feature_38   -0.888184
feature_80   -0.894393
feature_9    -0.951082
feature_78   -0.986292
feature_2    -1.592737
Length: 100, dtype: float64


### Step 5:Fit Ridge

In [108]:
from sklearn.linear_model import Ridge
ridge = Ridge()
ridge.fit(X_train , y_train)

Ridge()

**- Inspect Ridge coefficients**

In [109]:
ridge_coef = pd.Series(ridge.coef_ , index = X.columns)

In [110]:
print(ridge_coef.sort_values(ascending = False ))

feature_1     1.667151
feature_3     1.597605
feature_25    0.986113
feature_89    0.867631
feature_32    0.774896
                ...   
feature_38   -0.884623
feature_80   -0.885760
feature_9    -0.937433
feature_78   -0.964584
feature_2    -1.563243
Length: 100, dtype: float64


### Step 6: Fit Lasso

In [111]:
from sklearn.linear_model import Lasso
lasso = Lasso() #use smaller alpha
lasso.fit(X_train , y_train)

Lasso()

**- Inspect Lasso coefficients**

In [112]:
lasso_coef = pd.Series(lasso.coef_ , index = X.columns)
print(lasso_coef.sort_values(ascending = False))

feature_3      6.085621
feature_1      3.615025
feature_25     0.121160
feature_32     0.084945
feature_28     0.000000
                 ...   
feature_96    -0.000000
feature_99     0.000000
feature_100   -0.000000
feature_9     -0.060780
feature_2     -2.694853
Length: 100, dtype: float64


In [101]:
lasso.intercept_

np.float64(2.080217591832258)

In [102]:
print("Non-zero coefficients:")
print((lasso.coef_ != 0).sum())

Non-zero coefficients:
0


### Step 7 : Compare All Coefficients

In [113]:
coef_df = pd.DataFrame({
    "OLS": ols.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_
}, index=X.columns)

coef_df


,OLS,Ridge,Lasso
feature_1,1.699411,1.667151,3.615025
feature_2,-1.592737,-1.563243,-2.694853
feature_3,1.616039,1.597605,6.085621
feature_4,0.035080,0.031653,-0.000000
feature_5,0.322152,0.320134,0.000000
...,...,...,...
feature_96,-0.693287,-0.684743,-0.000000
feature_97,-0.111734,-0.115418,-0.000000
feature_98,0.353232,0.341622,0.000000
feature_99,0.188928,0.191875,0.000000


### Step 8: compare performance

In [100]:
from sklearn.metrics import mean_squared_error

models = {
    "OLS": ols,
    "Ridge": ridge,
    "Lasso": lasso
}

for name, model in models.items():

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    train_mse = mean_squared_error(y_train, train_pred)
    test_mse = mean_squared_error(y_test, test_pred)

    print(f"\n{name}")
    print(f"Train MSE: {train_mse:.2f}")
    print(f"Test MSE : {test_mse:.2f}")


OLS
Train MSE: 0.00
Test MSE : 87.60

Ridge
Train MSE: 18.18
Test MSE : 92.29

Lasso
Train MSE: 85.57
Test MSE : 115.31
